# P2 DAG-ERC-F Seed 42 — Nautilus Jupyter

This notebook runs the canonical P2 DAG-ERC-F baseline on Nautilus.

It is designed for a persistent Jupyter volume rather than Google Colab:

- no `google.colab` imports;
- no `/content` paths;
- no browser upload prompts;
- results and cache remain on the mounted volume;
- the final results ZIP is exposed as a clickable Jupyter link.

Before running, place these two files in the same persistent folder as this notebook, or set their exact paths in the configuration cell:

1. `p2_dagerc(1).zip` or another `p2_dagerc*.zip`
2. `IEMOCAP_features.pkl`


## 1. Configure paths and run options


In [ ]:
from pathlib import Path
import os

# Set this to the persistent directory containing this notebook and uploaded files.
# Path.cwd() is usually correct when the notebook was opened from that directory.
PERSISTENT_ROOT = Path.cwd().resolve()

# Set exact paths here only if auto-detection does not find the correct files.
P2_ZIP_PATH = None
IEMOCAP_PKL_PATH = None

# Output locations on the persistent volume.
EXTRACT_ROOT = PERSISTENT_ROOT / "p2_nautilus_workspace"
OUTPUT_ROOT = PERSISTENT_ROOT / "p2_nautilus_outputs"
CACHE_DIR = OUTPUT_ROOT / "cache"
FULL_OUTPUT_DIR = OUTPUT_ROOT / "full"
RESULT_ZIP = PERSISTENT_ROOT / "p2_dagerc_seed42_results.zip"

# Set these to False when resuming and you want to skip a completed stage.
RUN_SMOKE_TEST = True
RUN_CACHE = True
RUN_TRAIN = True
RUN_EVAL = True

print("Persistent root:", PERSISTENT_ROOT)
print("Output root:", OUTPUT_ROOT)


## 2. Locate files, extract the package, install dependencies, and verify the GPU


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile
import shutil

search_roots = [
    PERSISTENT_ROOT,
    Path.home(),
    Path("/workspace"),
    Path("/home/jovyan"),
    Path("/mnt/data"),
]

def find_candidates(pattern: str):
    found = []
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(pattern):
                try:
                    rp = p.resolve()
                except Exception:
                    rp = p
                if rp not in seen:
                    found.append(rp)
                    seen.add(rp)
        except PermissionError:
            pass
    return sorted(found)

if P2_ZIP_PATH is None:
    zip_candidates = find_candidates("p2_dagerc*.zip")
else:
    zip_candidates = [Path(P2_ZIP_PATH).expanduser().resolve()]

if IEMOCAP_PKL_PATH is None:
    pkl_candidates = [
        p for p in find_candidates("*.pkl")
        if "iemocap" in p.name.lower()
    ]
else:
    pkl_candidates = [Path(IEMOCAP_PKL_PATH).expanduser().resolve()]

print("P2 ZIP candidates:")
for p in zip_candidates:
    print(" ", p)

print("\nIEMOCAP PKL candidates:")
for p in pkl_candidates:
    print(" ", p)

if len(zip_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one P2 ZIP. Set P2_ZIP_PATH in the configuration cell. "
        f"Found: {zip_candidates}"
    )

if len(pkl_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one IEMOCAP PKL. Set IEMOCAP_PKL_PATH in the configuration cell. "
        f"Found: {pkl_candidates}"
    )

zip_path = zip_candidates[0]
data_path = pkl_candidates[0]

if not zip_path.is_file():
    raise FileNotFoundError(zip_path)
if not data_path.is_file():
    raise FileNotFoundError(data_path)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

package_marker = EXTRACT_ROOT / ".extracted_from"
needs_extract = True
if package_marker.exists():
    needs_extract = package_marker.read_text().strip() != str(zip_path)

if needs_extract:
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(EXTRACT_ROOT)
    package_marker.write_text(str(zip_path))
    print("Extracted:", zip_path)
else:
    print("Using existing extracted package:", EXTRACT_ROOT)

matches = [
    p for p in EXTRACT_ROOT.rglob("p2_dagerc")
    if p.is_dir() and (p / "run.py").exists()
]

if len(matches) != 1:
    raise RuntimeError(
        "Could not uniquely locate a p2_dagerc folder containing run.py. "
        f"Found: {matches}"
    )

WORKDIR = matches[0].resolve()
DATA_PATH = data_path.resolve()

os.chdir(WORKDIR)

print("\nWorking directory:", WORKDIR)
print("Dataset:", DATA_PATH)
print("Cache directory:", CACHE_DIR)
print("Full output directory:", FULL_OUTPUT_DIR)

subprocess.run(["nvidia-smi"], check=False)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.48.3",
    "scikit-learn",
    "pyyaml",
    "sentencepiece",
])

import torch
print("\nTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Start this notebook in a Nautilus GPU pod."
    )
print("GPU:", torch.cuda.get_device_name(0))
print("Setup complete.")


## 3. Run the implementation smoke test

This should complete quickly. Do not continue to the full run if it fails.


In [ ]:
import subprocess
import sys

if RUN_SMOKE_TEST:
    subprocess.run(
        [sys.executable, "smoke_test.py"],
        cwd=WORKDIR,
        check=True,
    )
else:
    print("Smoke test skipped by configuration.")


## 4. Cache frozen RoBERTa features

This stage can take time, but its output is persistent. If the kernel or pod restarts after this stage, set `RUN_CACHE = False` and reuse the saved cache.


In [ ]:
import subprocess
import sys

CACHE_DIR.mkdir(parents=True, exist_ok=True)

if RUN_CACHE:
    cmd = [
        sys.executable, "run.py",
        "--mode", "cache",
        "--data_path", str(DATA_PATH),
        "--config", "config.yaml",
        "--cache_dir", str(CACHE_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORKDIR, check=True)
else:
    print("Cache stage skipped. Existing cache:", CACHE_DIR)

print("Cache contents:")
for p in sorted(CACHE_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(CACHE_DIR), p.stat().st_size)


## 5. Train P2 on the full canonical split

The output is written directly to the persistent volume. Keep the browser session open while the cell runs. If Nautilus allows the pod to continue after browser disconnection, the cell may continue, but do not rely on that unless your deployment is configured for it.


In [ ]:
import subprocess
import sys

FULL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_TRAIN:
    cmd = [
        sys.executable, "run.py",
        "--mode", "train",
        "--data_path", str(DATA_PATH),
        "--config", "config.yaml",
        "--cache_dir", str(CACHE_DIR),
        "--output_dir", str(FULL_OUTPUT_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORKDIR, check=True)
else:
    print("Training skipped. Existing output:", FULL_OUTPUT_DIR)

best_model = FULL_OUTPUT_DIR / "best.pt"
if not best_model.exists():
    raise FileNotFoundError(
        f"Training did not produce the expected checkpoint: {best_model}"
    )

print("Best model:", best_model)
print("Size:", best_model.stat().st_size, "bytes")


## 6. Evaluate the selected checkpoint on the canonical test set


In [ ]:
import subprocess
import sys

predictions_path = FULL_OUTPUT_DIR / "predictions.json"
best_model = FULL_OUTPUT_DIR / "best.pt"

if RUN_EVAL:
    cmd = [
        sys.executable, "run.py",
        "--mode", "eval",
        "--data_path", str(DATA_PATH),
        "--config", "config.yaml",
        "--cache_dir", str(CACHE_DIR),
        "--model_path", str(best_model),
        "--split", "test",
        "--save_path", str(predictions_path),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORKDIR, check=True)
else:
    print("Evaluation skipped. Existing predictions:", predictions_path)

if not predictions_path.exists():
    raise FileNotFoundError(predictions_path)

print("Predictions saved:", predictions_path)


## 7. Verify results and create a downloadable ZIP

This verifies the canonical 1,592 test predictions and creates a ZIP on the persistent volume. The final line displays a clickable download link in Jupyter.


In [ ]:
import json
import shutil
from pathlib import Path
from IPython.display import FileLink, display

predictions_path = FULL_OUTPUT_DIR / "predictions.json"

with predictions_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

rows = (
    data["predictions"]
    if isinstance(data, dict) and "predictions" in data
    else data
)

print("Prediction rows:", len(rows))
assert len(rows) == 1592, (
    f"Expected 1592 canonical test predictions, got {len(rows)}"
)

if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()

archive_base = str(RESULT_ZIP.with_suffix(""))
archive_path = Path(
    shutil.make_archive(
        archive_base,
        "zip",
        root_dir=FULL_OUTPUT_DIR,
    )
)

print("Created:", archive_path)
print("Archive size:", archive_path.stat().st_size, "bytes")
display(FileLink(str(archive_path)))


## Resume notes

- If caching completed but training did not, set `RUN_CACHE = False`.
- If training completed and only evaluation remains, set `RUN_CACHE = False`, `RUN_TRAIN = False`, and keep `RUN_EVAL = True`.
- Do not delete `p2_nautilus_outputs/cache` between stages.
- The final archive is `p2_dagerc_seed42_results.zip` in the persistent notebook directory.
